# Preliminary Results

Questions:
1. Do macroeconomic and news signals move 1–3 months before WARN layoff spikes?
2. Do news features improve prediction beyond macro variables alone?
3. Can layoffs be grouped into low / medium / high risk months?

In [ ]:
# pip install statsmodels


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm


In [ ]:
master = pd.read_csv("../data/processed/master_monthly.csv")
master["month"] = pd.to_datetime(master["month"])

os.makedirs("../data/processed/figures", exist_ok=True)

master.head()

In [ ]:
# Main target over time
plt.figure(figsize=(12, 5))
plt.plot(master["month"], master["warn_layoffs"])
plt.title("Monthly California WARN Layoffs")
plt.xlabel("Month")
plt.ylabel("Affected employees")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../data/processed/figures/01_warn_layoffs_over_time.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Standardized overlay for lead-lag visual inspection
overlay_cols = [
    "warn_layoffs_z",
    "ca_unemployment_rate_z",
    "indeed_job_postings_index_z",
    "news_volume_z"
]

plt.figure(figsize=(12, 5))
for col in overlay_cols:
    plt.plot(master["month"], master[col], label=col)

plt.title("Standardized Time Series Overlay")
plt.xlabel("Month")
plt.ylabel("Z-score")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../data/processed/figures/02_standardized_overlay.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# WARN vs news tone
plt.figure(figsize=(12, 5))
plt.plot(master["month"], master["warn_layoffs_z"], label="warn_layoffs_z")
plt.plot(master["month"], master["news_tone_z"], label="news_tone_z")
plt.title("WARN Layoffs vs News Tone (Standardized)")
plt.xlabel("Month")
plt.ylabel("Z-score")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../data/processed/figures/03_warn_vs_news_tone.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Lead-lag correlation table: correlation of WARN at month t with predictor at t-lag

def build_lag_corr_table(df, target, predictors, max_lag=3):
    rows = []
    for predictor in predictors:
        for lag in range(0, max_lag + 1):
            corr = df[target].corr(df[predictor].shift(lag))
            rows.append({
                "predictor": predictor,
                "lag_months": lag,
                "correlation_with_warn": corr,
                "abs_correlation": abs(corr) if pd.notna(corr) else np.nan
            })
    return pd.DataFrame(rows)

predictors = [
    "ca_unemployment_rate",
    "fed_funds_rate",
    "indeed_job_postings_index",
    "news_volume",
    "news_tone"
]

lag_corr = build_lag_corr_table(master, "warn_layoffs", predictors, max_lag=3)
lag_corr

In [ ]:
best_lags = (
    lag_corr.sort_values(["predictor", "abs_correlation"], ascending=[True, False])
    .groupby("predictor", as_index=False)
    .first()
    .sort_values("abs_correlation", ascending=False)
)

best_lags

In [ ]:
heatmap_data = lag_corr.pivot(index="predictor", columns="lag_months", values="correlation_with_warn")

plt.figure(figsize=(8, 4))
sns.heatmap(heatmap_data, annot=True, cmap="coolwarm", center=0)
plt.title("Lead-Lag Correlation Heatmap")
plt.xlabel("Lag in months")
plt.ylabel("Predictor")
plt.tight_layout()
plt.savefig("../data/processed/figures/04_lag_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Regression: macro only vs macro + news
# Using lag1 predictors as a simple first-pass model

model_df = master.copy()

macro_features = [
    "ca_unemployment_rate_lag1",
    "fed_funds_rate_lag1",
    "indeed_job_postings_index_lag1"
]

full_features = macro_features + [
    "news_volume_log1p_lag1",
    "news_tone_lag1"
]

target = "warn_layoffs_log1p"

reg_df = model_df.dropna(subset=[target] + full_features).copy()

X1 = sm.add_constant(reg_df[macro_features])
X2 = sm.add_constant(reg_df[full_features])
y = reg_df[target]

model_macro = sm.OLS(y, X1).fit()
model_full = sm.OLS(y, X2).fit()

comparison = pd.DataFrame({
    "model": ["Macro only", "Macro + news"],
    "r_squared": [model_macro.rsquared, model_full.rsquared],
    "adj_r_squared": [model_macro.rsquared_adj, model_full.rsquared_adj],
    "aic": [model_macro.aic, model_full.aic],
    "bic": [model_macro.bic, model_full.bic]
})

comparison

In [ ]:
print(model_macro.summary())

In [ ]:
print(model_full.summary())

In [ ]:
# Simple out-of-sample comparison using the last 12 months as test data

split_date = pd.Timestamp("2024-07-01")

def time_split_rmse(df, features, target, split_date):
    temp = df.dropna(subset=features + [target]).copy()
    
    train = temp[temp["month"] < split_date].copy()
    test = temp[temp["month"] >= split_date].copy()
    
    model = LinearRegression()
    model.fit(train[features], train[target])
    
    pred = model.predict(test[features])
    rmse = np.sqrt(mean_squared_error(test[target], pred))
    
    return rmse, len(train), len(test)

rmse_macro, n_train_macro, n_test_macro = time_split_rmse(reg_df, macro_features, target, split_date)
rmse_full, n_train_full, n_test_full = time_split_rmse(reg_df, full_features, target, split_date)

oos_compare = pd.DataFrame({
    "model": ["Macro only", "Macro + news"],
    "test_rmse": [rmse_macro, rmse_full],
    "train_rows": [n_train_macro, n_train_full],
    "test_rows": [n_test_macro, n_test_full]
})

oos_compare

In [ ]:
# Risk classification feasibility
# Split WARN layoffs into 3 groups

master["risk_level"] = pd.qcut(
    master["warn_layoffs"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

risk_summary = (
    master.groupby("risk_level")[
        ["warn_layoffs", "ca_unemployment_rate", "indeed_job_postings_index", "news_volume", "news_tone"]
    ]
    .mean()
    .round(2)
)

risk_summary

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=master, x="risk_level", y="news_volume")
plt.title("News Volume by WARN Risk Level")
plt.xlabel("Risk level")
plt.ylabel("News volume")
plt.tight_layout()
plt.savefig("../data/processed/figures/05_news_volume_by_risk_level.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
comparison.to_csv("../data/processed/model_comparison.csv", index=False)
oos_compare.to_csv("../data/processed/oos_comparison.csv", index=False)
best_lags.to_csv("../data/processed/best_lags.csv", index=False)
risk_summary.to_csv("../data/processed/risk_summary.csv")

These are just preliminary results. The models use a small monthly sample, so the findings should jsut be interpreted as directional evidence rather than final causal conclsions.